# Avatar Chatbot v0.1

## Projekt: CV-basierter Chatbot

In diesem Notebook implementieren wir einen einfachen Chatbot, der Fragen zu einem Lebenslauf beantwortet.

### Anforderungen
- **Test 1**: Wenn eine Information im CV vorhanden ist → gib die korrekte Antwort
- **Test 2**: Wenn eine Information NICHT im CV vorhanden ist → antworte: "Das weiss ich leider nicht"

### Architektur
```
Prompt (mit CV im System-Kontext) 
    → LLM (gpt-4o-mini, temperature=0.0)
    → StrOutputParser
```

Basierend auf dem Pattern aus **M04a_LangChain101.ipynb** (Simple Chain mit LCEL-Syntax)

## 1. Setup & Environment

In [2]:
#@title 🔧 Umgebung einrichten (LOCAL VERSION)
# LOKAL: genai_lib muss bereits installiert sein
# Falls nicht: pip install -e /Users/wagnerg/Development/playground/GenAI_GW/lessons/GenAI/04_modul

import subprocess
import sys

# python-dotenv sicherstellen
try:
    from dotenv import load_dotenv
except ImportError:
    print("📦 Installiere python-dotenv...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

import os

# API Keys aus .env laden
env_path = '/Users/wagnerg/Development/playground/GenAI_GW/.env'
load_dotenv(env_path)

# Imports
from genai_lib.utilities import check_environment, mprint

print("✅ Umgebung wird vorbereitet...")
print()
check_environment()
print()
print(f"✓ OPENAI_API_KEY gesetzt: {'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY'] != ''}")
print(f"✓ genai_lib importiert erfolgreich")

✅ Umgebung wird vorbereitet...

Python Version: 3.13.2 (main, Feb  4 2025, 14:51:09) [Clang 16.0.0 (clang-1600.0.26.6)]

Installierte LangChain-Bibliotheken:
langchain                                1.1.0
langchain-chroma                         1.0.0
langchain-classic                        1.0.0
langchain-community                      0.4.1
langchain-core                           1.1.0
langchain-ollama                         1.0.0
langchain-openai                         1.1.0
langchain-text-splitters                 1.0.0

✓ OPENAI_API_KEY gesetzt: True
✓ genai_lib importiert erfolgreich


## 2. CV laden

In [3]:
def load_cv(filepath):
    """
    Lädt den Lebenslauf aus einer Markdown-Datei.
    
    Args:
        filepath: Pfad zur CV.md-Datei
        
    Returns:
        str: Inhalt des Lebenslaufs, oder None bei Fehler
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            cv_content = f.read()
        print(f"✅ CV erfolgreich geladen ({len(cv_content)} Zeichen)")
        return cv_content
    except FileNotFoundError:
        print(f"❌ Fehler: Datei nicht gefunden: {filepath}")
        return None
    except Exception as e:
        print(f"❌ Fehler beim Laden: {e}")
        return None

# Lade CV von verschiedenen möglichen Pfaden
cv_paths = [
    # 1. Im selben Verzeichnis wie das Notebook
    "CV.md",
    # 2. Im tasks/Avatar Verzeichnis (Absolute Path)
    "/Users/wagnerg/Development/playground/GenAI_GW/tasks/Avatar/CV.md",
]

cv_content = None
for cv_path in cv_paths:
    cv_content = load_cv(cv_path)
    if cv_content is not None:
        print(f"✅ CV geladen von: {cv_path}\n")
        break

if cv_content is None:
    print("❌ CV konnte in keinem der Standard-Pfade gefunden werden")

✅ CV erfolgreich geladen (2221 Zeichen)
✅ CV geladen von: CV.md



In [4]:
# Zeige CV-Preview
if cv_content:
    print("\n📄 CV-Vorschau (erste 600 Zeichen):\n")
    print(cv_content[:600])
    print("...\n")
    print(f"✅ Gesamte CV geladen und bereit für Chatbot")
else:
    print("❌ CV konnte nicht geladen werden")


📄 CV-Vorschau (erste 600 Zeichen):

# Lebenslauf: Max Mustermann

## Persönliche Daten

- **Name**: Max Mustermann
- **Geburtsdatum**: 15. März 1990
- **Wohnort**: München, Deutschland
- **E-Mail**: <max.mustermann@example.com>

## Berufliche Zusammenfassung

Erfahrener KI-Ingenieur mit 8+ Jahren Expertise in Machine Learning und Natural Language Processing.

## Berufserfahrung

### Senior KI-Ingenieur | TechCorp GmbH

**Zeitraum**: Januar 2020 - Heute
**Ort**: München, Deutschland

**Aufgaben**:

- Entwicklung von LLM-basierten Chatbots
- Leitung eines 5-köpfigen Teams
- Implementierung von RAG-Systemen mit LangChain

**Technol
...

✅ Gesamte CV geladen und bereit für Chatbot


## 3. Chatbot-Konfiguration

### Imports

In [5]:
# LangChain Imports
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.string import StrOutputParser

print("✅ LangChain Imports erfolgreich")

✅ LangChain Imports erfolgreich


### System-Prompt (kritisch für "Das weiss ich leider nicht")

In [6]:
# Definiere den System-Prompt mit strikten Regeln
# Diese Regeln sind KRITISCH für Test 2 ("Das weiss ich leider nicht")

system_prompt_template = """Du bist ein hilfreicher Assistent, der Fragen über eine Person beantwortet.

WICHTIGE REGELN:
1. Beantworte Fragen NUR auf Basis der bereitgestellten Lebenslauf-Informationen
2. Wenn eine Information NICHT im Lebenslauf vorhanden ist, antworte GENAU mit: "Das weiss ich leider nicht"
3. Erfinde KEINE Informationen - sei ehrlich, wenn du etwas nicht weisst
4. Antworte auf Deutsch
5. Sei präzise und konkret in deinen Antworten
6. Zitiere keine Markdown-Formatierung in deiner Antwort

LEBENSLAUF:
{cv_content}
"""

print("✅ System-Prompt definiert")

✅ System-Prompt definiert


### LangChain Chain zusammenbauen

In [7]:
# 1. Prompt-Template mit CV im Kontext
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_template),  # System-Prompt mit CV
    ("human", "{frage}")                # Benutzerfrage
])

# 2. Modell initialisieren
# temperature=0.0 ist WICHTIG: deterministische Antworten, keine Kreativität
llm = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    temperature=0.0  # Deterministische Antworten
)

# 3. Output Parser - konvertiert LLM-Output zu String
parser = StrOutputParser()

# 4. Chain zusammenbauen mit LCEL-Syntax (Pipe-Operator |)
# Muster aus M04a_LangChain101.ipynb
cv_chatbot_chain = prompt | llm | parser

print("✅ LangChain Chain erfolgreich erstellt")
print(f"   Chain: prompt | llm | parser")
print(f"   Modell: gpt-4o-mini (temperature=0.0)")

✅ LangChain Chain erfolgreich erstellt
   Chain: prompt | llm | parser
   Modell: gpt-4o-mini (temperature=0.0)


## 4. Interaktiver Test

In [8]:
# Hilfsfunktion für formatierte Ausgabe
def ask_chatbot(question):
    """
    Stelle eine Frage an den Chatbot.
    
    Args:
        question: Die Frage (deutscher Text)
        
    Returns:
        str: Die Antwort des Chatbots
    """
    if cv_content is None:
        return "❌ CV nicht geladen"
    
    try:
        # Invoke the chain
        answer = cv_chatbot_chain.invoke({
            "cv_content": cv_content,
            "frage": question
        })
        return answer
    except Exception as e:
        return f"❌ Fehler: {e}"

print("✅ ask_chatbot() Funktion bereit")

✅ ask_chatbot() Funktion bereit


In [9]:
# Versuche manuelle Fragen
print("="*70)
print("INTERAKTIVER TEST: Manuelle Fragen")
print("="*70)

# Beispiel-Fragen
test_questions = [
    "Wo hat Max studiert?",
    "Welches Auto fährt Max?",  # Sollte "Das weiss ich leider nicht" sein
]

for i, question in enumerate(test_questions, 1):
    print(f"\n❓ Frage {i}: {question}")
    answer = ask_chatbot(question)
    print(f"💬 Antwort: {answer}")
    print("-"*70)

INTERAKTIVER TEST: Manuelle Fragen

❓ Frage 1: Wo hat Max studiert?
💬 Antwort: Max hat an der Technischen Universität München und an der Ludwig-Maximilians-Universität München studiert.
----------------------------------------------------------------------

❓ Frage 2: Welches Auto fährt Max?
💬 Antwort: Das weiss ich leider nicht.
----------------------------------------------------------------------


## 5. Automatisierte Test-Suite

### Test-Funktion

In [10]:
def run_test(question, test_type):
    """
    Führt einen Test durch und überprüft das Ergebnis.
    
    Args:
        question: Die zu stellende Frage
        test_type: "available" (Info sollte im CV sein) oder "unavailable" (Info sollte NICHT im CV sein)
        
    Returns:
        bool: True wenn Test bestanden, False wenn fehlgeschlagen
    """
    if cv_content is None:
        print("❌ CV nicht geladen")
        return False
    
    answer = ask_chatbot(question)
    
    print(f"❓ Frage: {question}")
    print(f"💬 Antwort: {answer}")
    
    # Überprüfe ob die Antwort dem erwarteten Verhalten entspricht
    if test_type == "available":
        # Die Antwort sollte NICHT "Das weiss ich leider nicht" sein
        if "Das weiss ich leider nicht" in answer:
            print("❌ FEHLER: Information sollte im CV sein, aber Antwort ist 'Das weiss ich leider nicht'")
            return False
        else:
            print("✅ BESTANDEN: Bot findet die Information im CV")
            return True
    
    elif test_type == "unavailable":
        # Die Antwort sollte GENAU "Das weiss ich leider nicht" sein (oder ähnlich)
        if "Das weiss ich leider nicht" in answer:
            print("✅ BESTANDEN: Bot antwortet korrekt mit 'Das weiss ich leider nicht'")
            return True
        else:
            print("❌ FEHLER: Bot sollte 'Das weiss ich leider nicht' antworten")
            return False

print("✅ run_test() Funktion definiert")

✅ run_test() Funktion definiert


### Test 1: Informationen im CV vorhanden

In [ ]:
print("\n" + "="*70)
print("TEST 1: Informationen IM CV vorhanden")
print("Erwartet: Bot beantwortet Fragen korrekt basierend auf CV")
print("="*70)

# Test 1: Fragen mit Informationen im CV
test1_questions = [
    "Wo hat Max studiert?",
    "Welche Programmiersprachen beherrscht Max?",
    "Bei welchen Unternehmen hat Max gearbeitet?",
    "Welche Zertifikate hat Max?",
    "Wo wohnt Max?",
]

test1_results = []
for i, question in enumerate(test1_questions, 1):
    print(f"\n[Test 1.{i}]")
    result = run_test(question, "available")
    test1_results.append(result)
    print("-"*70)

print(f"\n📊 Test 1 Ergebnis: {sum(test1_results)}/{len(test1_results)} bestanden")


TEST 1: Informationen IM CV vorhanden
Erwartet: Bot beantwortet Fragen korrekt basierend auf CV

[Test 1.1]
❓ Frage: Wo hat Max studiert?
💬 Antwort: Max hat an der Technischen Universität München und an der Ludwig-Maximilians-Universität München studiert.
✅ BESTANDEN: Bot findet die Information im CV
----------------------------------------------------------------------

[Test 1.2]
❓ Frage: Welche Programmiersprachen beherrscht Max?
💬 Antwort: Max beherrscht die Programmiersprachen Python (Expert), SQL (Fortgeschritten) und hat Grundkenntnisse in JavaScript.
✅ BESTANDEN: Bot findet die Information im CV
----------------------------------------------------------------------

[Test 1.3]
❓ Frage: Bei welchen Unternehmen hat Max gearbeitet?
💬 Antwort: Max hat bei folgenden Unternehmen gearbeitet:

1. TechCorp GmbH
2. DataAnalytics AG
3. StartupX
✅ BESTANDEN: Bot findet die Information im CV
----------------------------------------------------------------------

[Test 1.4]
❓ Frage: Welche 

### Test 2: Informationen NICHT im CV vorhanden

In [12]:
print("\n" + "="*70)
print("TEST 2: Informationen NICHT IM CV")
print("Erwartet: Bot antwortet 'Das weiss ich leider nicht'")
print("="*70)

# Test 2: Fragen mit Informationen NICHT im CV
test2_questions = [
    "Ist Max verheiratet?",
    "Welches Auto fährt Max?",
    "Hat Max Kinder?",
    "Wie hoch ist Max' Gehalt?",
    "Spricht Max Französisch?",
]

test2_results = []
for i, question in enumerate(test2_questions, 1):
    print(f"\n[Test 2.{i}]")
    result = run_test(question, "unavailable")
    test2_results.append(result)
    print("-"*70)

print(f"\n📊 Test 2 Ergebnis: {sum(test2_results)}/{len(test2_results)} bestanden")


TEST 2: Informationen NICHT IM CV
Erwartet: Bot antwortet 'Das weiss ich leider nicht'

[Test 2.1]
❓ Frage: Ist Max verheiratet?
💬 Antwort: Das weiss ich leider nicht.
✅ BESTANDEN: Bot antwortet korrekt mit 'Das weiss ich leider nicht'
----------------------------------------------------------------------

[Test 2.2]
❓ Frage: Welches Auto fährt Max?
💬 Antwort: Das weiss ich leider nicht.
✅ BESTANDEN: Bot antwortet korrekt mit 'Das weiss ich leider nicht'
----------------------------------------------------------------------

[Test 2.3]
❓ Frage: Hat Max Kinder?
💬 Antwort: Das weiss ich leider nicht.
✅ BESTANDEN: Bot antwortet korrekt mit 'Das weiss ich leider nicht'
----------------------------------------------------------------------

[Test 2.4]
❓ Frage: Wie hoch ist Max' Gehalt?
💬 Antwort: Das weiss ich leider nicht.
✅ BESTANDEN: Bot antwortet korrekt mit 'Das weiss ich leider nicht'
----------------------------------------------------------------------

[Test 2.5]
❓ Frage: Spricht 

### Gesamtergebnis

In [ ]:
print("\n" + "="*70)
print("GESAMTERGEBNIS - ALL TESTS")
print("="*70)

total_tests = len(test1_results) + len(test2_results)
total_passed = sum(test1_results) + sum(test2_results)

print(f"\n✅ Test 1 (Info vorhanden):     {sum(test1_results)}/{len(test1_results)} bestanden")
print(f"✅ Test 2 (Info nicht vorhanden): {sum(test2_results)}/{len(test2_results)} bestanden")
print(f"\n🎯 GESAMT: {total_passed}/{total_tests} Tests bestanden")

if total_passed == total_tests:
    print("\n🎉 ERFOLG! Alle Tests bestanden! Der Chatbot funktioniert korrekt.")
else:
    print(f"\n⚠️ {total_tests - total_passed} Test(s) fehlgeschlagen. Überprüfe den System-Prompt.")

print("="*70)

## 6. Erkenntnisse & Zusammenfassung

### Was funktioniert gut in v0.1

✅ **Einfachheit**: Simple Chain ohne komplexe Komponenten (Prompt → LLM → Parser)

✅ **Nachvollziehbar**: Klarer Datenfluss, leicht zu debuggen

✅ **Testbar**: Automatisierte Tests validieren beide Szenarien

✅ **Wartbar**: Nur der System-Prompt ist der "Tuning"-Punkt

✅ **Schnell**: Keine Vektordatenbank, keine komplexe Setup

### Limitierungen von v0.1

❌ **Context-Größe**: Funktioniert nur für kurze CVs (< 2000 Tokens)

❌ **Keine Konversation**: Kein Memory, jede Frage ist isoliert

❌ **Statischer Content**: CV wird bei jeder Anfrage mitgeschickt

❌ **Modell-abhängig**: Halluzinationen sind immer möglich (reduziert durch temperature=0.0 und strikter Prompt)

### Ideen für v0.2 und später

**Kurzfristig (v0.2)**:
1. Konversations-Memory für Follow-up-Fragen
2. Strukturierte Ausgabe mit Pydantic (`with_structured_output()`)
3. Batch-Anfragen für mehrere Fragen parallel
4. Few-Shot-Beispiele für bessere "unbekannt"-Antworten

**Mittelfristig (v0.3)**:
5. RAG-System mit ChromaDB für längere CVs
6. Multi-CV-Management für mehrere Personen
7. Gradio-UI für Web-Interface

**Langfristig**:
8. Avatar-Persönlichkeit: Antwort-Stil lernen
9. Multimodal: CV mit Bild, Video
10. Fine-Tuning: Spezialisiertes Modell für diesen Use Case

### Learnings & Best Practices

**Prompt Engineering**:
- Explizite Regeln in GROSSBUCHSTABEN verstärken die Wichtigkeit
- Exakte Formulierung des gewünschten Outputs vorgeben
- Context direkt im System-Prompt für hohe Relevanz

**LangChain LCEL-Syntax**:
- Pipe-Operator `|` ist sehr lesbar
- Chain-Komponenten klar getrennt
- Leicht erweiterbar (z.B. + Retriever, + Memory)

**Temperature & Determinismus**:
- temperature=0.0 für konsistente Antworten
- Wichtig für zuverlässiges Verhalten bei bestimmten Antworten

### Fazit

Der Avatar Chatbot v0.1 demonstriert erfolgreich:
1. Ein einfaches, wartbares Design mit LangChain
2. Korrekte Handhabung von bekannten Informationen
3. Angemessene Reaktion auf unbekannte Informationen
4. Ein reproduzierbares Testing-Pattern

Der nächste Schritt für eine Production-Ready Version wäre die Hinzufügung von Konversations-Memory und möglicherweise ein RAG-System für längere Lebenslauf-Dokumente.

---

## Projekt-Metadaten

- **Version**: 0.1
- **Status**: ✅ Funktional
- **Technologie**: LangChain 1.0+, OpenAI GPT-4o-mini
- **Pattern**: Simple Chain (LCEL-Syntax)
- **Dokumentation**: tasks/Avatar/Task.md
- **CV-Datei**: CV.md (dieser Ordner)
- **Next Steps**: Konversations-Memory, RAG-System, UI